# Combo 4 - Chakra teacher-student distillation

This Kaggle notebook trains a compact MobileNetV3 student from the multi-scale Chakra ViT teacher. Upload the teacher checkpoint as a Kaggle Dataset, or leave the checkpoint path empty to use the ImageNet-initialized teacher. The student is the deployment candidate for live video; the ViT remains the accuracy reference.

In [1]:
!git clone -q https://github.com/varshasrao58/chakramodel.git /kaggle/working/chakramodel
%cd /kaggle/working/chakramodel
!pip install -q timm==0.9.16 opencv-python-headless
import os, glob, shutil, subprocess
assert os.path.exists('/kaggle/input'), 'Run this notebook on Kaggle with GPU enabled'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode())

/kaggle/working/chakramodel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━ 1.1/2.2 MB 32.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 30.0 MB/s eta 0:00:00


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB



In [2]:
import os
import glob
import shutil
from pathlib import Path

# 1. Define target paths explicitly expected by the training script
target_images = Path('/kaggle/working/chakramodel/src/data/kvasir-seg/images')
target_masks = Path('/kaggle/working/chakramodel/src/data/kvasir-seg/masks')

target_images.mkdir(parents=True, exist_ok=True)
target_masks.mkdir(parents=True, exist_ok=True)

# 2. Define the root of your newly attached dataset
dataset_root = Path('/kaggle/input/datasets/gokulrocky/chakra-leakbench-v1')

if not dataset_root.exists():
    raise FileNotFoundError(f"Could not find dataset at {dataset_root}. Check the sidebar.")

# 3. Flatten and copy all images and masks from the nested folders
print("Consolidating datasets into working directory...")
for img_path in dataset_root.rglob('images/*.*'):
    if img_path.is_file():
        shutil.copy(img_path, target_images / img_path.name)
        
for mask_path in dataset_root.rglob('masks/*.*'):
    if mask_path.is_file():
        shutil.copy(mask_path, target_masks / mask_path.name)

# 4. Find the teacher checkpoint (if uploaded)
teacher_matches = glob.glob('/kaggle/input/**/*.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pth', recursive=True)
teacher = teacher_matches[0] if teacher_matches else ''

print(f'images={len(list(target_images.glob("*")))}')
print(f'masks={len(list(target_masks.glob("*")))}')
print(f'teacher={teacher or "ImageNet initialization"}')

Consolidating datasets into working directory...


images=6523
masks=2248
teacher=ImageNet initialization


In [3]:
%cd /kaggle/working/chakramodel
import subprocess, sys, os

# Set PYTHONPATH so the script can import shared repository modules
my_env = os.environ.copy()
my_env["PYTHONPATH"] = "/kaggle/working/chakramodel:/kaggle/working/chakramodel/src:" + my_env.get("PYTHONPATH", "")

# Setup training command (using only flags supported by train_transformer.py)
command = [
    sys.executable, 
    '/kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py', 
    '--epochs', '30', 
    '--batch-size', '8'
]

if teacher:
    command += ['--teacher-checkpoint', teacher]

subprocess.run(command, check=True, env=my_env)

/kaggle/working/chakramodel


=== ChakraTransformer Training (High-Accuracy Research Track) ===
Batch Size: 8, Epochs: 30, LR: 0.0001
Initializing Vision Transformer (ViT-Large)...
Model initialized on cuda. Loading dataset...


  Epoch [  1/30] | Loss: 0.9760 | Val Dice: 0.4362 | Val IoU: 0.3154 | LR: 0.000100


  *** New best Dice: 0.4362 -> saved to /kaggle/working/chakramodel/src/weights/chakra_transformer_best.pth


  Epoch [  2/30] | Loss: 0.9455 | Val Dice: 0.6155 | Val IoU: 0.4962 | LR: 0.000099


...

  Epoch [ 16/30] | Loss: 0.8686 | Val Dice: 0.7156 | Val IoU: 0.6102 | LR: 0.000045


  Epoch [ 17/30] | Loss: 0.8670 | Val Dice: 0.7854 | Val IoU: 0.6922 | LR: 0.000040


  *** New best Dice: 0.7854 -> saved to /kaggle/working/chakramodel/src/weights/chakra_transformer_best.pth


  Epoch [ 18/30] | Loss: 0.8649 | Val Dice: 0.7816 | Val IoU: 0.6875 | LR: 0.000035


  Epoch [ 19/30] | Loss: 0.8602 | Val Dice: 0.8012 | Val IoU: 0.7087 | LR: 0.000030


In [ ]:
!ls -l /kaggle/working/chakramodel/src/chakra_transformer/

In [ ]:
!mv /kaggle/working/chakramodel/data/kvasir-seg /kaggle/working/chakramodel/data/Kvasir-SEG

In [ ]:
!python /kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py --help

In [ ]:
!PYTHONPATH=/kaggle/working/chakramodel:/kaggle/working/chakramodel/src python /kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py --help

In [ ]:
!grep -rn "Kvasir-SEG" /kaggle/working/chakramodel/src/

In [ ]:
!sed -n '38,45p' /kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py

In [ ]:
!mkdir -p /kaggle/working/chakramodel/src/data
!mv /kaggle/working/chakramodel/data/Kvasir-SEG /kaggle/working/chakramodel/src/data/kvasir-seg

In [ ]:
!find /kaggle/working/chakramodel -name "train_transformer.py"

In [ ]:
!find /kaggle/working/chakramodel -name "*.py"

In [ ]:
%cd /kaggle/working/chakramodel
import subprocess, sys, os

# 1. Add the repository root to the Python path
my_env = os.environ.copy()
my_env["PYTHONPATH"] = "/kaggle/working/chakramodel:/kaggle/working/chakramodel/src:" + my_env.get("PYTHONPATH", "")

# 2. Setup the command
# 2. Setup the command
command = [
    sys.executable, 
    '/kaggle/working/chakramodel/ACTUAL_PATH_YOU_FOUND/train_transformer.py', # <-- UPDATE THIS LINE
    '--epochs', '30', 
    '--batch-size', '8'
]

if teacher:
    command += ['--teacher-checkpoint', teacher]

# 3. Run the command with the updated environment path
subprocess.run(command, check=True, env=my_env)

In [ ]:
%cd /kaggle/working/chakramodel
import subprocess, sys
command = [
    sys.executable, 
    '/kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py', 
    '--epochs', '30', 
    '--batch-size', '8', 
    '--grad-accum', '2', 
    '--workers', '2'
]
if teacher: command += ['--teacher-checkpoint', teacher]
subprocess.run(command, check=True)

In [ ]:
import torch, time
from src.chakra_transformer.student_segmenter import ChakraLiveStudent
student = ChakraLiveStudent().cuda().eval()
student.load_state_dict(torch.load('weights/combo4_student_best.pth', map_location='cuda'))
x = torch.randn(1, 3, 352, 352, device='cuda')
for _ in range(20): student(x)
torch.cuda.synchronize(); start = time.perf_counter()
for _ in range(100): student(x)
torch.cuda.synchronize(); elapsed = time.perf_counter() - start
print(f'parameters={sum(p.numel() for p in student.parameters()):,}; FPS={100 / elapsed:.2f}; latency_ms={elapsed * 10:.2f}')
shutil.copy('weights/combo4_student_best.pth', '/kaggle/working/combo4_student_best.pth')

In [ ]:
!rm -rf /kaggle/working/chakramodel
!git clone https://github.com/varshasrao58/chakramodel.git /kaggle/working/chakramodel

In [ ]:
!ls -l /kaggle/working/chakramodel/src/chakra_transformer/

In [ ]:
# 1. Safely change directory back to the main working folder
%cd /kaggle/working

# 2. Clean up any broken remnants
!rm -rf chakramodel

# 3. Clone the repository fresh
!git clone https://github.com/varshasrao58/chakramodel.git

In [ ]:
!ls -l /kaggle/working/chakramodel/src/chakra_transformer/